# Phase 2 (Step 4): Self-Reflective RAG (Self-RAG)

Standard RAG is blind. It retrieves documents and gives them to the LLM, hoping the documents actually contain the answer. If the retrieval is bad, the LLM hallucinates.

**Self-RAG** fixes this by adding a "Grader" step. After retrieving documents, an LLM acts as a judge and asks: *"Do these documents actually answer the user's question?"*
If yes, it generates the final answer. If no, it rewrites the search query and tries again.

We will build this using LangGraph to see how a State Graph handles this logic beautifully.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama

# 1. Define our State
# This holds the variables that get passed between nodes in our graph
class GraphState(TypedDict):
    question: str
    context: str
    generation: str
    re_tries: int

llm = ChatOllama(model="qwen2.5:7b-instruct", temperature=0)

c:\Users\sujat\projects\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Define our Nodes (Functions)
In LangGraph, nodes are just python functions that take the `state` as input and return an updated `state`.

In [2]:
def retrieve_node(state):
    print("\n--- [NODE: Retrieving Data] ---")
    question = state["question"]
    
    # Simulating a vector database search
    # If the user asks about 'PTO', we find it. Otherwise, we simulate a bad search result.
    if "pto" in question.lower():
        context = "Employees receive 20 days of Paid Time Off (PTO) per year."
    else:
        context = "The cafeteria serves pizza on Fridays."
        
    print(f"Retrieved Context: {context}")
    return {"context": context, "question": question}

def generate_node(state):
    print("\n--- [NODE: Generating Answer] ---")
    question = state["question"]
    context = state["context"]
    
    prompt = f"Use the following context to answer the question.\nContext: {context}\nQuestion: {question}\nAnswer:"
    response = llm.invoke(prompt)
    return {"generation": response.content}

def rewrite_query_node(state):
    print("\n--- [NODE: Rewriting Query] ---")
    question = state["question"]
    
    # Ask the LLM to rewrite the query to be more specific
    prompt = f"Rewrite this question to be a better search query for a database: {question}\nRewritten Query:"
    response = llm.invoke(prompt)
    print(f"New Query: {response.content.strip()}")
    
    # Increment retries so we don't get stuck in an infinite loop
    retries = state.get("re_tries", 0) + 1
    
    return {"question": response.content.strip(), "re_tries": retries}

### 2. The Conditional Edge (The Grader)
This is the magic of Self-RAG. This function decides where the graph goes next.

In [4]:
def grade_documents(state):
    print("\n--- [EDGE: Grading Documents] ---")
    question = state["question"]
    context = state["context"]
    retries = state.get("re_tries", 0)
    
    if retries >= 2:
        print("Max retries reached. Forcing generation.")
        return "generate"

    prompt = f"""
    You are a grader assessing relevance of a retrieved document to a user question.
    Document: {context}
    Question: {question}
    Does the document contain keyword(s) or semantic meaning related to the question?
    Respond strictly with 'yes' or 'no'.
    """
    response = llm.invoke(prompt)
    grade = response.content.strip().lower()
    
    if "yes" in grade:
        print("Grade: Relevant! Moving to generation.")
        return "generate"
    else:
        print("Grade: NOT relevant! Moving to rewrite.")
        return "rewrite"

### 3. Compile the Graph
We tie the nodes and edges together.

In [5]:
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)
workflow.add_node("rewrite", rewrite_query_node)

# Build Graph
workflow.set_entry_point("retrieve")

# After retrieving, we grade. It either goes to generate or rewrite.
workflow.add_conditional_edges(
    "retrieve",
    grade_documents,
    {
        "generate": "generate",
        "rewrite": "rewrite"
    }
)

# After rewriting, we always try retrieving again
workflow.add_edge("rewrite", "retrieve")

# After generating, we are done
workflow.add_edge("generate", END)

app = workflow.compile()

### 4. Test it!
Watch what happens when we ask about PTO (it finds it immediately) vs when we ask about Insurance (it fails, rewrites the query, tries again, and eventually gives up).

In [6]:
print("========== TEST 1: Good Query ==========")
inputs = {"question": "How much PTO do I get?", "re_tries": 0}
result = app.invoke(inputs)
print("\nFinal Answer:\n", result["generation"])

print("\n\n========== TEST 2: Bad Query ==========")
inputs = {"question": "What is our insurance policy?", "re_tries": 0}
result = app.invoke(inputs)
print("\nFinal Answer:\n", result["generation"])

========== TEST 1: Good Query ==========

--- [NODE: Retrieving Data] ---
Retrieved Context: Employees receive 20 days of Paid Time Off (PTO) per year.

--- [EDGE: Grading Documents] ---
Grade: Relevant! Moving to generation.

--- [NODE: Generating Answer] ---

Final Answer:
 You get 20 days of Paid Time Off (PTO) per year.


========== TEST 2: Bad Query ==========

--- [NODE: Retrieving Data] ---
Retrieved Context: The cafeteria serves pizza on Fridays.

--- [EDGE: Grading Documents] ---
Grade: NOT relevant! Moving to rewrite.

--- [NODE: Rewriting Query] ---
New Query: What are the details of our current insurance policy?

--- [NODE: Retrieving Data] ---
Retrieved Context: The cafeteria serves pizza on Fridays.

--- [EDGE: Grading Documents] ---
Grade: NOT relevant! Moving to rewrite.

--- [NODE: Rewriting Query] ---
New Query: What are the details of our current insurance policy number [Policy Number]?

--- [NODE: Retrieving Data] ---
Retrieved Context: The cafeteria serves pizza on